In [1]:
import os
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0\\research'

In [2]:
os.chdir("../")
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class FederatedLearningConfig:
    root_dir: Path
    client_models_dir: Path
    aggregated_model_path: Path
    client_data_dir: Path
    base_model_path: Path
    selected_features_path: Path
    params_image_size: list
    params_batch_size: int
    params_fl_batch_size: int
    params_classes: int
    params_learning_rate: float
    num_clients: int
    fl_rounds: int
    local_epochs: int

In [4]:
from ThyroidCancer.constants import *
from ThyroidCancer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_federated_learning_config(self) -> FederatedLearningConfig:
        config = self.config.federated_training
        fl_config = self.config.federated_learning
        prepare_base_model_config = self.config.prepare_base_model
        feature_selection_config = self.config.feature_selection
        
        create_directories([config.root_dir, config.client_models_dir])

        federated_learning_config = FederatedLearningConfig(
            root_dir=Path(config.root_dir),
            client_models_dir=Path(config.client_models_dir),
            aggregated_model_path=Path(config.aggregated_model_path),
            client_data_dir=Path(fl_config.client_data_dir),
            base_model_path=Path(prepare_base_model_config.updated_base_model_path),
            selected_features_path=Path(feature_selection_config.selected_features_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_fl_batch_size=self.params.FL_BATCH_SIZE,
            params_classes=self.params.CLASSES,
            params_learning_rate=self.params.LEARNING_RATE,
            num_clients=self.params.NUM_CLIENTS,
            fl_rounds=self.params.FL_ROUNDS,
            local_epochs=self.params.LOCAL_EPOCHS
        )

        return federated_learning_config

In [6]:
import tensorflow as tf
import numpy as np
import pickle
from ThyroidCancer import logger
from pathlib import Path

In [7]:
class FederatedLearning:
    def __init__(self, config: FederatedLearningConfig):
        self.config = config
        self.client_models = []
        
    def _build_model(self):
        """Build the EfficientNetB0 model architecture"""
        base_model = tf.keras.applications.EfficientNetB0(
            input_shape=self.config.params_image_size,
            weights='imagenet',
            include_top=False
        )
        for layer in base_model.layers[:-20]:
            layer.trainable = False
        for layer in base_model.layers[-20:]:
            layer.trainable = True
        
        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(base_model.output)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5, name="top_dropout")(x)
        predictions = tf.keras.layers.Dense(
            units=self.config.params_classes,
            activation="softmax",
            name="predictions"
        )(x)

        model = tf.keras.models.Model(
            inputs=base_model.input,
            outputs=predictions,
            name="EfficientNetB0_FL"
        )
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.config.params_learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
        
        return model
    
    def initialize_global_model(self):
        """Initialize the global model with pretrained weights"""
        logger.info("Initializing global model...")
        self.global_model = self._build_model()
        
        # Load pretrained weights from prepare_base_model
        logger.info(f"Loading pretrained weights from {self.config.base_model_path}...")
        self.global_model.load_weights(self.config.base_model_path)
        logger.info("Global model initialized successfully.")
        
    def get_client_data_generators(self, client_id):
        """Create data generator for a specific client"""
        client_dir = self.config.client_data_dir / f"client_{client_id}"
        
        datagen = tf.keras.preprocessing.image.ImageDataGenerator(
            rescale=1./255,
            validation_split=0.2
        )
        
        train_generator = datagen.flow_from_directory(
            directory=client_dir,
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_fl_batch_size,
            class_mode='categorical',
            subset='training',
            shuffle=True
        )
        
        val_generator = datagen.flow_from_directory(
            directory=client_dir,
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_fl_batch_size,
            class_mode='categorical',
            subset='validation',
            shuffle=False
        )
        
        return train_generator, val_generator
    
    def train_client(self, client_id, round_num):
        """Train a single client's model"""
        logger.info(f"\n{'='*60}")
        logger.info(f"Training Client {client_id} - Round {round_num}")
        logger.info(f"{'='*60}")
        
        # Create client model and load global weights
        client_model = self._build_model()
        client_model.set_weights(self.global_model.get_weights())
        
        # Get client data
        train_gen, val_gen = self.get_client_data_generators(client_id)
        
        # Train locally
        history = client_model.fit(
            train_gen,
            epochs=self.config.local_epochs,
            validation_data=val_gen,
            verbose=1
        )
        
        # Save client model weights
        client_weights_path = self.config.client_models_dir / f"client_{client_id}_round_{round_num}.h5"
        client_model.save_weights(client_weights_path)
        logger.info(f"Client {client_id} weights saved to {client_weights_path}")
        
        return client_model.get_weights(), history
    
    def federated_averaging(self, client_weights_list):
        """Aggregate client weights using FedAvg"""
        logger.info("\nPerforming Federated Averaging...")
        
        # Initialize averaged weights
        avg_weights = []
        
        # Average each layer's weights
        for layer_idx in range(len(client_weights_list[0])):
            layer_weights = np.array([client_weights[layer_idx] for client_weights in client_weights_list])
            avg_layer_weights = np.mean(layer_weights, axis=0)
            avg_weights.append(avg_layer_weights)
        
        logger.info("Federated averaging completed.")
        return avg_weights
    
    def run_federated_learning(self):
        """Run the complete federated learning process"""
        logger.info("\n" + "="*80)
        logger.info("STARTING FEDERATED LEARNING")
        logger.info(f"Number of Clients: {self.config.num_clients}")
        logger.info(f"FL Rounds: {self.config.fl_rounds}")
        logger.info(f"Local Epochs per Round: {self.config.local_epochs}")
        logger.info("="*80 + "\n")
        
        # Initialize global model
        self.initialize_global_model()
        
        # Federated Learning Rounds
        for round_num in range(1, self.config.fl_rounds + 1):
            logger.info(f"\n{'#'*80}")
            logger.info(f"FEDERATED LEARNING ROUND {round_num}/{self.config.fl_rounds}")
            logger.info(f"{'#'*80}\n")
            
            client_weights_list = []
            
            # Train each client
            for client_id in range(1, self.config.num_clients + 1):
                client_weights, history = self.train_client(client_id, round_num)
                client_weights_list.append(client_weights)
            
            # Aggregate weights
            aggregated_weights = self.federated_averaging(client_weights_list)
            
            # Update global model
            self.global_model.set_weights(aggregated_weights)
            logger.info(f"Global model updated for round {round_num}.")
        
        # Save final aggregated model
        logger.info(f"\nSaving final aggregated model to {self.config.aggregated_model_path}...")
        self.global_model.save_weights(self.config.aggregated_model_path)
        logger.info("Federated Learning completed successfully!")
        logger.info("="*80)

In [ ]:
try:
    config = ConfigurationManager()
    fl_config = config.get_federated_learning_config()
    fl = FederatedLearning(config=fl_config)
    
    fl.run_federated_learning()
    
except Exception as e:
    raise e